# 00 - Dataset Preparation & Structuring Injection

## 1. Introduction and Reproducibility
This notebook establishes the foundational dataset for the AegisAML research pipeline. We leverage the **IBM AMLSim** simulator to generate a baseline transaction graph and explicitly inject mathematically controlled structuring (smurfing) typologies. This guarantees a verified ground-truth subset for evaluating the temporal graph model's recall.



In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import time
from datetime import datetime, timedelta
from pathlib import Path

# 1.1 Reproducibility Configuration
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Visualization settings
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Initialize Directories
DIRS = ['../data/raw', '../data/processed', '../reports', '../figures']
for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)
    
start_time = time.time()
print(f"Notebook execution started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")



## 2. AMLSim Overview & Configuration
IBM AMLSim generates synthetic transaction networks based on specified parameters. To ensure our temporal graph model has sufficient positive samples of structuring rings, we dynamically modify the parameter configuration.

### 2.1 Parameter Modifications

| Parameter | Default | Modified | Reason |
| :--- | :---: | :---: | :--- |
| `fan_in` | 0.02 | 0.30 | Drastically increase smurfing probability to ensure adequate positive samples. |
| `fan_out` | 0.02 | 0.10 | Increase layering/distribution networks. |
| `cycle` | 0.05 | 0.10 | Increase circular transaction flows. |



In [ ]:
def configure_amlsim():
    # If the param.json exists (assuming AMLSim is cloned), we patch it.
    config_path = Path('AMLSim/paramFiles/1K/param.json')
    if config_path.exists():
        with open(config_path, 'r') as f:
            config = json.load(f)
            
        # Apply modifications
        if 'alert_patterns' in config:
            config['alert_patterns']['fan_in']['ratio'] = 0.30
            config['alert_patterns']['fan_out']['ratio'] = 0.10
            config['alert_patterns']['cycle']['ratio'] = 0.10
            
        with open(config_path, 'w') as f:
            json.dump(config, f, indent=4)
        print("AMLSim configuration successfully patched.")
    else:
        print("AMLSim repository not detected locally. Skipping configuration patch.")

configure_amlsim()

# In a full run, we would execute the simulator here:
# import subprocess
# subprocess.run(["./build.sh"], cwd="AMLSim")
# subprocess.run(["./run.sh", "conf/param.json"], cwd="AMLSim")



## 3. Dataset Generation & Statistics
For the purpose of this notebook's immediate execution, if AMLSim outputs are not present, we simulate a statistically similar baseline dataset.



In [ ]:
def load_or_generate_base_dataset(num_accounts=1000, num_txs=10000):
    tx_path = Path('AMLSim/outputs/transactions.csv')
    acct_path = Path('AMLSim/outputs/accounts.csv')
    
    if tx_path.exists() and acct_path.exists():
        print("Loading AMLSim outputs...")
        tx_df = pd.read_csv(tx_path)
        acct_df = pd.read_csv(acct_path)
    else:
        print("AMLSim outputs not found. Generating baseline dataset...")
        accounts = range(1, num_accounts + 1)
        acct_df = pd.DataFrame({'account_id': accounts, 'customer_id': accounts, 'init_balance': 5000})
        
        senders = np.random.choice(accounts, size=num_txs)
        receivers = np.random.choice(accounts, size=num_txs)
        # Fix self-transfers
        receivers = np.where(senders == receivers, (receivers % num_accounts) + 1, receivers)
        
        base_time = datetime(2023, 1, 1)
        timestamps = [base_time + timedelta(hours=np.random.randint(0, 24*30)) for _ in range(num_txs)]
        
        tx_df = pd.DataFrame({
            'tx_id': [f"tx_{i}" for i in range(num_txs)],
            'sender_id': senders,
            'receiver_id': receivers,
            'amount': np.random.lognormal(mean=4.0, sigma=1.0, size=num_txs).round(2),
            'timestamp': timestamps,
            'is_sar': 0,
            'typology': 'normal'
        })
    return tx_df, acct_df

tx_df, acct_df = load_or_generate_base_dataset()

# Compute Dataset Statistics
stats = {
    "Total Accounts": len(acct_df),
    "Total Transactions": len(tx_df),
    "Date Range": f"{tx_df['timestamp'].min()} to {tx_df['timestamp'].max()}",
    "Average Transaction Size": f"${tx_df['amount'].mean():.2f}"
}
stats_df = pd.DataFrame(list(stats.items()), columns=["Metric", "Value"])
display(stats_df)



## 4. Synthetic Typology Injection (Structuring)

### Algorithm 1: Temporal Structuring Injection
**Input**: Base transactions $G$, set of accounts $V$, threshold $	au$, max days $\Delta d$, number of rings $R$
**Output**: Augmented transactions $G'$, Typology Manifest $M$

For each ring $r \in \{1 \dots R\}$:
1. Select a destination account $T \in V$.
2. Sample $N \in [4, 12]$ mule accounts $S \subset V \setminus \{T\}$.
3. Define total illicit amount $A \sim U(0.85	au, 0.99	au)$.
4. Partition $A$ into $N$ transactions such that $orall x \in A_{parts}, x < 	au$.
5. Assign timestamps $t_i$ strictly increasing within a $\Delta d$ window.
6. Record ring metadata into manifest $M$.



In [ ]:
def inject_structuring_rings(tx_df, acct_df, num_rings=100, threshold=10000, max_days=5):
    injected_txs = []
    manifest = []
    
    accounts = acct_df['account_id'].values
    
    for r_idx in range(num_rings):
        target = np.random.choice(accounts)
        num_mules = np.random.randint(4, 13)
        mules = np.random.choice(accounts[accounts != target], size=num_mules, replace=False)
        
        # Total amount just under reporting threshold
        total_amount = np.random.uniform(threshold * 0.85, threshold * 0.99)
        
        # Random partition of the amount
        partitions = np.random.dirichlet(np.ones(num_mules)) * total_amount
        
        # Temporal window
        start_time = datetime(2023, 1, 1) + timedelta(days=np.random.randint(0, 20))
        
        ring_txs = []
        for i, mule in enumerate(mules):
            tx_time = start_time + timedelta(hours=np.random.randint(0, max_days*24))
            ring_txs.append({
                'tx_id': f"ring_{r_idx}_{i}",
                'sender_id': mule,
                'receiver_id': target,
                'amount': round(partitions[i], 2),
                'timestamp': tx_time,
                'is_sar': 1,
                'typology': 'structuring'
            })
            
        # Ensure temporal ordering for realism
        ring_txs.sort(key=lambda x: x['timestamp'])
        injected_txs.extend(ring_txs)
        
        manifest.append({
            'Ring_ID': f"R{r_idx:03d}",
            'Typology': 'Structuring',
            'Accounts_Involved': num_mules + 1,
            'Duration_Days': max_days,
            'Total_Amount': round(total_amount, 2),
            'Ground_Truth': 'Positive'
        })
        
    inj_df = pd.DataFrame(injected_txs)
    manifest_df = pd.DataFrame(manifest)
    
    return pd.concat([tx_df, inj_df], ignore_index=True), manifest_df

augmented_tx_df, manifest_df = inject_structuring_rings(tx_df, acct_df)

print(f"Successfully injected {len(manifest_df)} structuring rings.")
display(manifest_df.head())



## 5. Topological Visualization
Visualizing multiple rings to confirm the `fan_in` bipartite structure.



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Visual Verification of Injected Structuring Rings (Fan-In Topology)", fontsize=16)

for idx, r_id in enumerate(['R000', 'R001', 'R002']):
    ring_txs = augmented_tx_df[augmented_tx_df['tx_id'].str.startswith(f"ring_{int(r_id[1:])}_")]
    
    G = nx.DiGraph()
    for _, row in ring_txs.iterrows():
        G.add_edge(row['sender_id'], row['receiver_id'], weight=row['amount'])
        
    pos = nx.spring_layout(G, seed=RANDOM_SEED)
    nx.draw(G, pos, ax=axes[idx], with_labels=True, node_color='#aed6f1', node_size=1500, edge_color='gray', arrows=True)
    axes[idx].set_title(f"Ring {r_id}")

plt.savefig('../figures/ring_topologies.png')
plt.show()



## 6. Export and Metadata Footer
Exporting the `typology_manifest.csv`, augmented data, and finalizing execution.



In [ ]:
# Exporting Data
augmented_tx_df.to_parquet('../data/raw/transactions.parquet', index=False)
acct_df.to_parquet('../data/raw/accounts.parquet', index=False)
manifest_df.to_csv('../data/raw/typology_manifest.csv', index=False)

# Execution Metadata
end_time = time.time()
execution_time = end_time - start_time
print("--- Notebook Metadata ---")
print(f"Dataset Version: v1.0 (AMLSim Augmented)")
print(f"Execution Time: {execution_time:.2f} seconds")
print("Artifacts successfully exported to ../data/raw/")

